<a href="https://colab.research.google.com/github/janithcyapa/Statistical-Learning-e20452/blob/main/Assignments/Assignment_07_Bayesian_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Assignment : Bayesian Inference**
### ME526 : Introduction to Statistical Learning
---
> YAPA W.S.P.Y.J.C.
>
> E/20/452
---

In [5]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm, beta

## **Q1. Bayesian Estimation of a User Ability Parameter from Item Responses**

An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:

$$Y_i=
\begin{cases}
1, & \text{if the user answers item } i \text{ correctly},\\
0, & \text{if the user answers item } i \text{ incorrectly}.
\end{cases}$$

The platform assumes that the probability of a correct response is governed by a two-parameter logistic (2PL) item response model. Specifically, conditional on the user's latent ability parameter $\Theta=\theta$, the response probability for item $i$ is:

$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $a_i>0$ is the known discrimination parameter, and $b_i$ is the known difficulty parameter of item $i$.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed responses** up to the current step $k$ (where $1 \le k \le n$).

Before observing any responses, the platform initializes the user's latent ability estimate with a standard normal prior distribution:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right) \quad \text{implying} \quad \Theta \sim \mathscr{N}(0,1).$$

As the user progresses, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$.


### Tasks

1. **Visualizing the Mechanics:** Plot $P(Y_i=1\mid \Theta=\theta)$ vs $\theta$ using Plotly for two distinct values of $a_i$, where one of those $a_i$ values is paired with three different difficulty values of $b_i$. Interpret how moving $b_i$ shifts the curve horizontally.

In [2]:
# @title
theta = np.linspace(-4, 4, 200)

# 2PL model
def prob_correct(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

fig = go.Figure()

# a = 2.0, b = (-1, 0, 1)
for b in [-1, 0, 1]:
    fig.add_trace(go.Scatter(
        x=theta,
        y=prob_correct(theta, a=2.0, b=b),
        mode='lines',
        name=f'a=2.0, b={b}'
    ))

# a = 0.8, b = 0
fig.add_trace(go.Scatter(
    x=theta,
    y=prob_correct(theta, a=0.8, b=0),
    mode='lines',
    line=dict(dash='dash', color='white'),
    name='a=0.8, b=0'
))

fig.update_layout(
    title="2PL IRT Mode",
    xaxis_title="Latent Ability",
    yaxis_title="Probability of Correct Response",
    template="plotly_dark"
)

fig.show()

2. **Sequential Likelihood Contribution:** Write down the likelihood contribution $L(y_k \mid \theta)$ of a *single* new response $y_k$ at step $k$, given the latent ability $\theta$. Then, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

Assuming responses are conditionally independent and $\Theta = \theta$, the likelihood contribution of a single new response $y_k$ is a Bernoulli likelihood.

$$L(y_k \mid \theta) = p_k(\theta)^{y_k} (1 - p_k(\theta))^{1 - y_k}$$

history vector $\mathbf{y}^{(k)} = (y_1, \dots, y_k)$,
the joint likelihood ,
$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k p_i(\theta)^{y_i} (1 - p_i(\theta))^{1 - y_i}$$

3. **Mathematical Formulation of the Running Update:** Write down the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, up to a proportionality constant, using the prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ and the new observation $y_k$.

By Bayes' theorem,
$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$
\
Substituting the 2PL likelihood,

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[ p_k(\theta)^{y_k} (1 - p_k(\theta))^{1 - y_k} \right] f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

4. **Dynamic Shifting:** Explain how a correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) mathematically shifts the peak of the running posterior density distribution relative to the previous step.

Mathematically, when we multiply the running prior density by this monotonically increasing likelihood function, it heavily suppresses the probability mass on the left side (low $\theta$) while preserving or boosting mass on the right side (high $\theta$). After normalization, this inevitably shifts the peak (the Maximum A Posteriori estimate) of the running posterior density toward larger values of $\theta$.

5. **Tracking Certainty and Sharpness:** Explain how the discrimination parameter $a_k$ of the current item alters the variance (or "sharpness") of the distribution during a running update. What happens when $a_k$ is very large versus very small?

- Large $a_k$ -  The logistic curve transitions very sharply from 0 to 1 near $b_k$. This creates a steep likelihood function that aggressively cuts off probability mass on the incorrect side of $b_k$. Multiplying by this reduces the variance of the running posterior and system gains high certainty.

- Small $a_k$ - The logistic curve is very flat. The likelihood function is almost uniform. Updating the posterior with this barely alters the variance or the shape of the density.  

6. **Numerical Implementation of a Running Grid:** Describe a algorithmic approach to numerically approximate and maintain this running posterior density function on a fixed grid of $\theta$-values. Explicitly state how you would perform the sequential normalization step computationally after an item is answered.


1. Grid Initialization - Create a fixed array of $\theta$ values. Initialize the prior array by evaluating the standard normal density $\mathscr{N}(0, 1)$ at each grid point.  

2. Likelihood Evaluation - When response $y_k$ arrives, evaluate the likelihood $L(y_k \mid \theta)$ at every point on the $\theta$ grid.  

3. Multiplication - Multiply the prior array by the likelihood array element-wise to get the unnormalized posterior array.  

4. Normalization - Use numerical integration across the grid to calculate the total area under the unnormalized posterior curve. Divide the entire unnormalized array by this area scalar to ensure it integrates to 1, making it a valid density.  

5. Recursion - Set this normalized array as the new prior for step $k+1$.  





7. **Evaluating Convergence over the Timeline:** Suppose the user's true, hidden latent ability is $\theta_{\text{true}} = 0.75$. Write a Python script that extends your previous grid simulation to track the performance of the running estimators over a sequence of $n = 20$ items.
* **Simulate Responses:** Dynamically generate the user's responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against the true response probability $p_k(\theta_{\text{true}})$. Give each item a random difficulty $b_k \sim \mathscr{N}(0, 1)$ and a random discrimination $a_k \sim \text{Uniform}(0.5, 2.0)$.
* **Track Estimators:** At each step $k$, calculate and store the running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) and the running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) estimate.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $20$. Add a static horizontal reference line at $y = 0.75$ representing $\theta_{\text{true}}$.
* **Analysis:** Briefly explain how the distance between your estimators and $\theta_{\text{true}}$ changes as $k$ increases, and interpret what this implies about the platform's confidence in its measurement.

In [4]:
# @title
np.random.seed(42)
n_items = 20
theta_true = 0.75

# numerical grid
grid_points = 1000
theta_grid = np.linspace(-5, 5, grid_points)

# Prior
posterior_density = norm.pdf(theta_grid, loc=0, scale=1)

# tracking estimators
bayes_means = [np.trapezoid(theta_grid * posterior_density, theta_grid)]
map_estimates = [theta_grid[np.argmax(posterior_density)]]

for k in range(n_items):
    a_k = np.random.uniform(0.5, 2.0)
    b_k = np.random.normal(0, 1)

    # Calculate true probability and simulate the user's response
    p_true = 1 / (1 + np.exp(-a_k * (theta_true - b_k)))
    y_k = 1 if np.random.rand() < p_true else 0

    # Evaluate Likelihood
    p_grid = 1 / (1 + np.exp(-a_k * (theta_grid - b_k)))
    likelihood = p_grid if y_k == 1 else (1 - p_grid)

    # Unnormalized Posterior
    unnormalized_post = posterior_density * likelihood

    # Sequential Normalization
    area = np.trapezoid(unnormalized_post, theta_grid)
    posterior_density = unnormalized_post / area

    current_bayes_mean = np.trapezoid(theta_grid * posterior_density, theta_grid)
    current_map = theta_grid[np.argmax(posterior_density)]

    bayes_means.append(current_bayes_mean)
    map_estimates.append(current_map)

fig = go.Figure()

fig.add_trace(go.Scatter(
    y=bayes_means,
    mode='lines+markers',
    name='Posterior Mean',
    line=dict(color='#00ffcc', width=2)
))

fig.add_trace(go.Scatter(
    y=map_estimates,
    mode='lines+markers',
    name='MAP Estimate',
    line=dict(color='#ff00ff', width=2)
))

fig.add_hline(
    y=theta_true,
    line_dash="dash",
    line_color="white",
    annotation_text="True Latent Ability",
    annotation_position="bottom right"
)

fig.update_layout(
    title="atent Ability Estimators",
    xaxis_title="Item Number (k)",
    yaxis_title="Estimated Latent Ability",
    template="plotly_dark",
    xaxis=dict(tick0=0, dtick=2),
    legend=dict(x=0.02, y=0.98)
)

fig.show()

## **Q2.  Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates**

An e-commerce platform wants to optimize its recommendation engine by dynamically estimating the click-through rate (CTR) of a newly launched advertisement. Since user traffic arrives continuously, the platform updates its belief about the advertisement's performance **one impression at a time** rather than waiting for large batch updates.

Let $\Theta = \theta$ represent the true, hidden conversion rate (probability of a click) of the advertisement, where $\theta \in [0, 1]$.

Let $Y_k$ denote a single user's interaction with the advertisement at time step $k$:

$$Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}, \\
0, & \text{if the user does not click the advertisement}.
\end{cases}$$

The platform assumes that conditional on the true conversion rate $\Theta = \theta$, each user interaction is an independent Bernoulli trial:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed user interactions** up to the current impression step $k$ (where $1 \le k \le n$).

Before observing any data, the platform assigns a flexible **Beta distribution** as the initial prior over the unknown parameter $\Theta$:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\mathrm{B}(\alpha_0, \beta_0)} \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1} \quad \text{implying} \quad \Theta \sim \text{Beta}(\alpha_0, \beta_0)$$

where $\mathrm{B}(\cdot, \cdot)$ is the Beta function acting as the normalizing constant. Under a sequential framework, the posterior distribution at step $k-1$ serves directly as the prior distribution for step $k$.


### **Tasks**

**1. Structural Probability and Properties**
Plot the probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution using Plotly for three distinct parameter pairs:

* Uninformative state: $(\alpha=1, \beta=1)$
* Right-skewed state: $(\alpha=2, \beta=8)$
* Left-skewed state: $(\alpha=8, \beta=2)$

Interpret how changing the balance between $\alpha$ and $\beta$ shifts the center of mass of the density function over the domain $[0, 1]$.

Changing the balance between $\alpha$ and $\beta$ mathematically shifts the expected value over the domain $[0, 1]$. Because the expected value of a Beta distribution is given by $\frac{\alpha}{\alpha + \beta}$, increasing $\alpha$ relative to $\beta$ shifts the center of mass to the right , whereas increasing $\beta$ shifts it to the left. An equal balance ($\alpha = \beta = 1$) indicates complete uniform uncertainty across all possible click-through rates

In [6]:
# @title
# Domain
theta = np.linspace(0, 1, 500)

params = [
    (1, 1, "Uninformative State (1, 1)"),
    (2, 8, "Right-Skewed State (2, 8)"),
    (8, 2, "Left-Skewed State (8, 2)")
]

fig = go.Figure()

for a, b, name in params:
    fig.add_trace(go.Scatter(
        x=theta,
        y=beta.pdf(theta, a, b),
        mode='lines',
        name=name,
        fill='tozeroy',
        opacity=0.6
    ))

fig.update_layout(
    title="Properties of the Beta Distribution Prior",
    xaxis_title="Theta",
    yaxis_title="Probability Density",
    template="plotly_dark"
)

fig.show()

**2. Sequential Likelihood and Joint History**

Write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* isolated response $y_k$ at step $k$, given the click probability $\theta$. Following this, express the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

Assuming independent Bernoulli trial,

The likelihood contribution of a single, isolated response $y_k$ ,
$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$

For the running history vector $\mathbf{y}^{(k)} = (y_1, \dots, y_k)$,

 the joint likelihood function is the product of the individual independent likelihoods,

  $$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k \theta^{y_i} (1 - \theta)^{1 - y_i} = \theta^{\sum_{i=1}^k y_i} (1 - \theta)^{k - \sum_{i=1}^k y_i}$$

**3. Closed-Form Analytical Updates (Conjugacy)**

Using Bayes' Theorem, derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$. Prove analytically that the posterior remains in the Beta family (**Beta-Binomial Conjugacy**) by explicitly writing down the closed-form update parameters $\alpha_k$ and $\beta_k$ as simple arithmetic updates of $\alpha_{k-1}$, $\beta_{k-1}$, and $y_k$. Also compute the **Posterior Mean** of the latent parameter $\Theta$ at time step $k$ (i.e. $\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]$).

By Bayes' Theorem in density form,

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

Substituting the Beta prior and the Bernoulli likelihood

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[ \theta^{y_k} (1 - \theta)^{1 - y_k} \right] \left[ \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right]$$


$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$

The functional form remains a Beta distribution. Closed-form arithmetic updates for the new parameters $\alpha_k = \alpha_{k-1} + y_k$$\beta_k = \beta_{k-1} + 1 - y_k$

at time step $k$,

 $$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k}$$

**4. Dynamic Shifting Mechanics**

Explain how an observed click ($y_k = 1$) vs. a non-click ($y_k = 0$) shifts the peak of the running density distribution mathematically. Contrast this analytical framework against non-conjugate setups (such as the 2PL IRT model) where numerical grid integration is strictly required.

- Observed Click ($y_k = 1$) - Mathematically, this increases $\alpha_k$ by 1 while leaving $\beta_k$ completely unchanged. The ratio $\frac{\alpha_k}{\alpha_k + \beta_k}$ increases,

- Observed Non-Click ($y_k = 0$) - This adds 1 to $\beta_k$ and leaves $\alpha_k$ unchanged. The ratio decreases, shifting the distribution's peak dynamically to the left.

- Conjugate vs. Non-Conjugate - Because the Beta prior and Binomial likelihood combine algebraically to create another Beta distribution, only need to store and update two scalars. In non-conjugate setup do not yield a recognizable functional form,the density must be approximated over a vast discrete grid and normalized via computational numerical integration at every step.  

**5. Running Point Estimators**

State the exact closed-form equations used to evaluate the following point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)

Equation,
$$\frac{\alpha_k}{\alpha_k + \beta_k}$$

And Condition is Always valid


* **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

$$\frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$$
$alpha_k > 1$ and $\beta_k > 1$




**6. Performance Tracking and Convergence Analysis**

Suppose the advertisement's true, hidden click-through rate is $\theta_{\text{true}} = 0.35$. Write a Python script to track the performance of your closed-form sequential estimators over a timeline of $n = 100$ impressions:

* **Initialize State:** Set the base prior parameters to $\alpha_0 = 1, \beta_0 = 1$ (representing uniform initial uncertainty).
* **Simulate Responses:** Dynamically generate user responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against $\theta_{\text{true}}$.
* **Track Estimators:** Loop through each step, update $\alpha_k$ and $\beta_k$ analytically, and store the computed values for $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $100$. Add a static horizontal reference line at $y = 0.35$ representing $\theta_{\text{true}}$.
* **Analysis:** Explain how the distance between your estimators and $\theta_{\text{true}}$ responds as the sampling size $k$ approaches $100$. What does this imply about the accumulation of evidence over time relative to the choice of the initial prior?

In [7]:
# @title
# State Init
np.random.seed(101)
n_impressions = 100
theta_true = 0.35

# (Beta(1,1))
alpha_k = 1
beta_k = 1

# Step 0
bayes_estimates = [alpha_k / (alpha_k + beta_k)]
map_estimates = [0.5]

for k in range(n_impressions):
    # user response based on True CTR
    y_k = 1 if np.random.rand() < theta_true else 0

    # Closed-Form Conjugate Update
    alpha_k += y_k
    beta_k += (1 - y_k)

    #  Running Point Estimators
    current_bayes = alpha_k / (alpha_k + beta_k)

    # (alpha > 1, beta > 1)
    if alpha_k > 1 and beta_k > 1:
        current_map = (alpha_k - 1) / (alpha_k + beta_k - 2)
    elif alpha_k > 1 and beta_k == 1:
        current_map = 1.0
    elif alpha_k == 1 and beta_k > 1:
        current_map = 0.0
    else:
        current_map = 0.5

    bayes_estimates.append(current_bayes)
    map_estimates.append(current_map)

fig = go.Figure()

fig.add_trace(go.Scatter(
    y=bayes_estimates,
    mode='lines+markers',
    name='Posterior Mean',
    line=dict(color='#00ffcc', width=2)
))

fig.add_trace(go.Scatter(
    y=map_estimates,
    mode='lines+markers',
    name='MAP Estimate',
    line=dict(color='#ff00ff', width=2)
))

fig.add_hline(
    y=theta_true,
    line_dash="dash",
    line_color="white",
    annotation_text="True Hidden CTR (0.35)",
    annotation_position="bottom right"
)

fig.update_layout(
    title="Dynamic Tracking of CTR Estimators via Conjugate Updates",
    xaxis_title="Impression Step (k)",
    yaxis_title="Estimated Click-Through Rate",
    template="plotly_dark",
    legend=dict(x=0.75, y=0.95)
)

fig.show()